# Use Case 04 — Manufacturing Defect Detection

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rajvermatx/careeralign.com/blob/main/gcp-mle/usecases/notebooks/04-defect-detection.ipynb)

**GCP MLE Use Case Series — CareerAlign.com**

Build an end-to-end image classification pipeline for manufacturing defect detection using:
- Synthetic defect image generation with NumPy
- Transfer learning with MobileNetV2
- Data augmentation for industrial imagery
- Per-class precision/recall evaluation
- Confusion matrix visualization
- Model export considerations for GCP deployment

---

## 1. Setup & Installations

In [ ]:
# Install required packages
!pip install -q tensorflow numpy matplotlib scikit-learn Pillow

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from PIL import Image, ImageDraw
import os
import warnings
warnings.filterwarnings('ignore')

print(f"TensorFlow version: {tf.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")

In [ ]:
# Configuration
IMG_SIZE = 128           # Image dimensions (128x128 for faster training)
NUM_CLASSES = 4          # normal, scratch, crack, dent
BATCH_SIZE = 32
EPOCHS_PHASE1 = 8        # Train head only
EPOCHS_PHASE2 = 12       # Fine-tune upper layers
SAMPLES_PER_CLASS = 300  # Synthetic images per class
SEED = 42

CLASS_NAMES = ['normal', 'scratch', 'crack', 'dent']

np.random.seed(SEED)
tf.random.set_seed(SEED)

## 2. Generate Synthetic Defect Images

In a real scenario, images come from production line cameras. Here we generate
synthetic images using NumPy to simulate:
- **Normal**: Clean metallic surface with slight texture
- **Scratch**: Linear marks across the surface
- **Crack**: Irregular branching lines
- **Dent**: Circular depressions on the surface

In [ ]:
def generate_base_surface(size=IMG_SIZE):
    """Generate a synthetic metallic surface texture."""
    # Base gray with slight gradient (simulates lighting)
    base = np.ones((size, size, 3), dtype=np.float32) * 0.55
    
    # Add subtle gradient (lighting direction)
    gradient = np.linspace(0.45, 0.65, size).reshape(1, -1, 1)
    base = base * gradient
    
    # Add fine texture noise (surface roughness)
    texture = np.random.normal(0, 0.03, (size, size, 3)).astype(np.float32)
    base = base + texture
    
    # Add very subtle horizontal machining lines
    for i in range(0, size, np.random.randint(3, 8)):
        if i < size:
            intensity = np.random.uniform(-0.02, 0.02)
            base[i, :, :] += intensity
    
    return np.clip(base, 0, 1)


def generate_normal(size=IMG_SIZE):
    """Generate a defect-free surface."""
    return generate_base_surface(size)


def generate_scratch(size=IMG_SIZE):
    """Generate a surface with scratch defect (linear marks)."""
    surface = generate_base_surface(size)
    img = Image.fromarray((surface * 255).astype(np.uint8))
    draw = ImageDraw.Draw(img)
    
    # Draw 1-3 scratches as lines
    num_scratches = np.random.randint(1, 4)
    for _ in range(num_scratches):
        # Random start and end points
        x1, y1 = np.random.randint(0, size, 2)
        # Scratches tend to be roughly linear with some direction
        angle = np.random.uniform(0, np.pi)
        length = np.random.randint(size // 3, size)
        x2 = int(x1 + length * np.cos(angle))
        y2 = int(y1 + length * np.sin(angle))
        
        width = np.random.randint(1, 3)
        brightness = np.random.randint(80, 140)  # Darker than surface
        color = (brightness, brightness, brightness + 10)
        draw.line([(x1, y1), (x2, y2)], fill=color, width=width)
    
    result = np.array(img).astype(np.float32) / 255.0
    return result


def generate_crack(size=IMG_SIZE):
    """Generate a surface with crack defect (irregular branching lines)."""
    surface = generate_base_surface(size)
    img = Image.fromarray((surface * 255).astype(np.uint8))
    draw = ImageDraw.Draw(img)
    
    # Start point for crack
    x, y = np.random.randint(size // 4, 3 * size // 4, 2)
    
    # Draw main crack with random walk
    points = [(x, y)]
    num_segments = np.random.randint(8, 20)
    for _ in range(num_segments):
        dx = np.random.randint(-12, 13)
        dy = np.random.randint(-12, 13)
        x = np.clip(x + dx, 0, size - 1)
        y = np.clip(y + dy, 0, size - 1)
        points.append((x, y))
    
    dark = np.random.randint(30, 80)
    if len(points) > 1:
        draw.line(points, fill=(dark, dark, dark), width=1)
    
    # Add 1-2 branching cracks
    num_branches = np.random.randint(1, 3)
    for _ in range(num_branches):
        if len(points) > 2:
            branch_start = points[np.random.randint(1, len(points) - 1)]
            bx, by = branch_start
            branch_pts = [branch_start]
            for _ in range(np.random.randint(3, 8)):
                bx = np.clip(bx + np.random.randint(-10, 11), 0, size - 1)
                by = np.clip(by + np.random.randint(-10, 11), 0, size - 1)
                branch_pts.append((bx, by))
            if len(branch_pts) > 1:
                draw.line(branch_pts, fill=(dark + 20, dark + 20, dark + 20), width=1)
    
    result = np.array(img).astype(np.float32) / 255.0
    return result


def generate_dent(size=IMG_SIZE):
    """Generate a surface with dent defect (circular depressions)."""
    surface = generate_base_surface(size)
    
    # Add 1-3 dents
    num_dents = np.random.randint(1, 4)
    for _ in range(num_dents):
        cx = np.random.randint(size // 4, 3 * size // 4)
        cy = np.random.randint(size // 4, 3 * size // 4)
        radius = np.random.randint(8, 25)
        
        # Create circular gradient (darker center, lighter edges)
        y_grid, x_grid = np.ogrid[:size, :size]
        dist = np.sqrt((x_grid - cx)**2 + (y_grid - cy)**2)
        mask = dist < radius
        
        # Depression effect: darker center with bright ring (specular highlight)
        depression = np.zeros((size, size), dtype=np.float32)
        depression[mask] = -0.15 * (1 - dist[mask] / radius)
        
        # Bright ring at edge of dent
        ring_mask = (dist >= radius * 0.8) & (dist < radius * 1.2)
        depression[ring_mask] = 0.08
        
        surface[:, :, 0] += depression
        surface[:, :, 1] += depression
        surface[:, :, 2] += depression
    
    return np.clip(surface, 0, 1)

In [ ]:
# Generate the full synthetic dataset
print("Generating synthetic defect images...")

generators = {
    0: generate_normal,     # normal
    1: generate_scratch,    # scratch
    2: generate_crack,      # crack
    3: generate_dent,       # dent
}

images = []
labels = []

for class_idx, gen_fn in generators.items():
    print(f"  Generating {SAMPLES_PER_CLASS} '{CLASS_NAMES[class_idx]}' images...")
    for _ in range(SAMPLES_PER_CLASS):
        img = gen_fn(IMG_SIZE)
        images.append(img)
        labels.append(class_idx)

images = np.array(images, dtype=np.float32)
labels = np.array(labels, dtype=np.int32)

print(f"\nDataset shape: {images.shape}")
print(f"Labels shape:  {labels.shape}")
print(f"Class distribution: {dict(zip(CLASS_NAMES, np.bincount(labels)))}")

In [ ]:
# Visualize sample images from each class
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Synthetic Manufacturing Defect Images', fontsize=16, fontweight='bold')

for class_idx in range(NUM_CLASSES):
    class_images = images[labels == class_idx]
    for row in range(2):
        sample_idx = np.random.randint(0, len(class_images))
        ax = axes[row, class_idx]
        ax.imshow(class_images[sample_idx])
        ax.set_title(CLASS_NAMES[class_idx].upper(), fontsize=12, fontweight='bold')
        ax.axis('off')

plt.tight_layout()
plt.show()

## 3. Build Image Dataset with Train/Val/Test Splits

In [ ]:
from sklearn.model_selection import train_test_split

# Stratified split: 70% train, 15% validation, 15% test
X_train, X_temp, y_train, y_temp = train_test_split(
    images, labels, test_size=0.3, random_state=SEED, stratify=labels
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=SEED, stratify=y_temp
)

print(f"Train set: {X_train.shape[0]} images")
print(f"Val set:   {X_val.shape[0]} images")
print(f"Test set:  {X_test.shape[0]} images")
print(f"\nTrain class distribution: {dict(zip(CLASS_NAMES, np.bincount(y_train)))}")
print(f"Val class distribution:   {dict(zip(CLASS_NAMES, np.bincount(y_val)))}")
print(f"Test class distribution:  {dict(zip(CLASS_NAMES, np.bincount(y_test)))}")

In [ ]:
# Compute class weights for imbalanced data handling
# (In real manufacturing data, 95%+ would be 'normal')
from sklearn.utils.class_weight import compute_class_weight

class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weight_dict = dict(enumerate(class_weights_array))
print("Class weights:", {CLASS_NAMES[k]: f"{v:.3f}" for k, v in class_weight_dict.items()})

## 4. Data Augmentation Pipeline

Manufacturing-specific augmentation:
- **Flips & rotations**: Parts can arrive at any orientation
- **Slight brightness/contrast**: Simulate lighting variation between shifts
- **Minimal color jitter**: Preserve discoloration signal
- **No heavy blur**: Would mask fine cracks

In [ ]:
# Build data augmentation layer
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.15),
    layers.RandomZoom((-0.1, 0.1)),
    layers.RandomBrightness(factor=0.1),
    layers.RandomContrast(factor=0.1),
], name="data_augmentation")

# Visualize augmentation effect
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
fig.suptitle('Data Augmentation Examples (same input image)', fontsize=14, fontweight='bold')

sample_image = X_train[0]
axes[0, 0].imshow(sample_image)
axes[0, 0].set_title('Original', fontweight='bold')
axes[0, 0].axis('off')

for i in range(1, 10):
    row, col = divmod(i, 5)
    augmented = data_augmentation(tf.expand_dims(sample_image, 0), training=True)
    axes[row, col].imshow(tf.squeeze(augmented).numpy())
    axes[row, col].set_title(f'Aug #{i}')
    axes[row, col].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Create tf.data pipelines
train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train))
train_ds = (
    train_ds
    .shuffle(1000, seed=SEED)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

val_ds = tf.data.Dataset.from_tensor_slices((X_val, y_val))
val_ds = val_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test))
test_ds = test_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

print(f"Train batches: {tf.data.experimental.cardinality(train_ds).numpy()}")
print(f"Val batches:   {tf.data.experimental.cardinality(val_ds).numpy()}")
print(f"Test batches:  {tf.data.experimental.cardinality(test_ds).numpy()}")

## 5. Transfer Learning with MobileNetV2

MobileNetV2 is ideal for manufacturing edge deployment:
- Only **3.4 MB** at int8 quantization (fits on Edge TPU)
- Fast inference (~20ms on Edge TPU)
- Good feature extraction from ImageNet pre-training

**Two-phase training strategy:**
1. Freeze backbone, train classification head only
2. Unfreeze top layers and fine-tune with lower learning rate

In [ ]:
def build_defect_model(num_classes=NUM_CLASSES, input_shape=(IMG_SIZE, IMG_SIZE, 3)):
    """Build MobileNetV2-based defect classifier with custom head."""
    
    # Load pre-trained MobileNetV2 (ImageNet weights)
    base_model = keras.applications.MobileNetV2(
        include_top=False,
        weights='imagenet',
        input_shape=input_shape,
    )
    
    # Freeze all backbone layers (Phase 1)
    base_model.trainable = False
    
    # Build the full model with augmentation + classification head
    inputs = keras.Input(shape=input_shape)
    
    # Data augmentation (only active during training)
    x = data_augmentation(inputs)
    
    # MobileNetV2 expects input in [-1, 1] range
    x = keras.applications.mobilenet_v2.preprocess_input(x * 255.0)
    
    # Feature extraction
    x = base_model(x, training=False)
    
    # Classification head
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    
    model = keras.Model(inputs, outputs, name='defect_classifier')
    return model, base_model


model, base_model = build_defect_model()
model.summary()

In [ ]:
# Phase 1: Train classification head only (backbone frozen)
print("=" * 60)
print("PHASE 1: Training classification head (backbone frozen)")
print("=" * 60)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

history_phase1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_PHASE1,
    class_weight=class_weight_dict,
    verbose=1,
)

print(f"\nPhase 1 final val accuracy: {history_phase1.history['val_accuracy'][-1]:.4f}")

In [ ]:
# Phase 2: Fine-tune top layers of backbone
print("=" * 60)
print("PHASE 2: Fine-tuning top layers of MobileNetV2")
print("=" * 60)

# Unfreeze the backbone
base_model.trainable = True

# Freeze all layers except the last 30
num_layers = len(base_model.layers)
fine_tune_at = max(0, num_layers - 30)
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

trainable_count = sum(1 for l in base_model.layers if l.trainable)
frozen_count = sum(1 for l in base_model.layers if not l.trainable)
print(f"Backbone layers: {num_layers} total, {trainable_count} trainable, {frozen_count} frozen")

# Recompile with lower learning rate for fine-tuning
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

# Early stopping to prevent overfitting
early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=4,
    restore_best_weights=True,
    verbose=1,
)

history_phase2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_PHASE2,
    class_weight=class_weight_dict,
    callbacks=[early_stop],
    verbose=1,
)

print(f"\nPhase 2 final val accuracy: {history_phase2.history['val_accuracy'][-1]:.4f}")

In [ ]:
# Plot training history for both phases
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Combine histories
acc = history_phase1.history['accuracy'] + history_phase2.history['accuracy']
val_acc = history_phase1.history['val_accuracy'] + history_phase2.history['val_accuracy']
loss = history_phase1.history['loss'] + history_phase2.history['loss']
val_loss = history_phase1.history['val_loss'] + history_phase2.history['val_loss']
epochs_range = range(1, len(acc) + 1)
phase1_end = EPOCHS_PHASE1

# Accuracy plot
axes[0].plot(epochs_range, acc, 'b-', label='Train Accuracy', linewidth=2)
axes[0].plot(epochs_range, val_acc, 'r-', label='Val Accuracy', linewidth=2)
axes[0].axvline(x=phase1_end, color='gray', linestyle='--', alpha=0.7, label='Fine-tune start')
axes[0].set_title('Model Accuracy', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss plot
axes[1].plot(epochs_range, loss, 'b-', label='Train Loss', linewidth=2)
axes[1].plot(epochs_range, val_loss, 'r-', label='Val Loss', linewidth=2)
axes[1].axvline(x=phase1_end, color='gray', linestyle='--', alpha=0.7, label='Fine-tune start')
axes[1].set_title('Model Loss', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Evaluation: Per-Class Precision/Recall & Confusion Matrix

In manufacturing defect detection, **recall is more important than precision**:
- False positive (good part flagged) = re-inspection cost (~$0.10)
- False negative (defect missed) = field failure cost (~$100-$10,000)

We need per-defect-type metrics, not just overall accuracy.

In [ ]:
# Generate predictions on the test set
y_pred_probs = model.predict(X_test, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)

# Overall test accuracy
test_loss, test_acc = model.evaluate(test_ds, verbose=0)
print(f"Test Loss:     {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")
print()

# Per-class classification report
print("=" * 60)
print("PER-CLASS CLASSIFICATION REPORT")
print("=" * 60)
report = classification_report(
    y_test, y_pred,
    target_names=CLASS_NAMES,
    digits=4
)
print(report)

In [ ]:
# Confusion Matrix
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Raw counts
cm = confusion_matrix(y_test, y_pred)
disp1 = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)
disp1.plot(ax=axes[0], cmap='Blues', values_format='d')
axes[0].set_title('Confusion Matrix (Counts)', fontsize=14, fontweight='bold')

# Normalized (percentages)
cm_norm = confusion_matrix(y_test, y_pred, normalize='true')
disp2 = ConfusionMatrixDisplay(confusion_matrix=cm_norm, display_labels=CLASS_NAMES)
disp2.plot(ax=axes[1], cmap='Oranges', values_format='.2%')
axes[1].set_title('Confusion Matrix (Normalized)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Per-class precision/recall bar chart
report_dict = classification_report(
    y_test, y_pred,
    target_names=CLASS_NAMES,
    output_dict=True
)

precisions = [report_dict[name]['precision'] for name in CLASS_NAMES]
recalls = [report_dict[name]['recall'] for name in CLASS_NAMES]
f1s = [report_dict[name]['f1-score'] for name in CLASS_NAMES]

x = np.arange(len(CLASS_NAMES))
width = 0.25

fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar(x - width, precisions, width, label='Precision', color='#3b82f6')
bars2 = ax.bar(x, recalls, width, label='Recall', color='#f59e0b')
bars3 = ax.bar(x + width, f1s, width, label='F1 Score', color='#22c55e')

ax.set_xlabel('Defect Type', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Per-Class Precision / Recall / F1', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([name.upper() for name in CLASS_NAMES])
ax.legend(fontsize=11)
ax.set_ylim(0, 1.1)
ax.grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{height:.2f}',
                    xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 3), textcoords="offset points",
                    ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

## 7. Visualize Predictions on Test Images

In [ ]:
# Show predictions on random test images
fig, axes = plt.subplots(3, 5, figsize=(18, 11))
fig.suptitle('Model Predictions on Test Images', fontsize=16, fontweight='bold')

indices = np.random.choice(len(X_test), 15, replace=False)

for i, idx in enumerate(indices):
    row, col = divmod(i, 5)
    ax = axes[row, col]
    
    ax.imshow(X_test[idx])
    
    true_label = CLASS_NAMES[y_test[idx]]
    pred_label = CLASS_NAMES[y_pred[idx]]
    confidence = y_pred_probs[idx][y_pred[idx]] * 100
    
    correct = y_test[idx] == y_pred[idx]
    color = 'green' if correct else 'red'
    symbol = 'OK' if correct else 'MISS'
    
    ax.set_title(
        f'True: {true_label}\nPred: {pred_label} ({confidence:.1f}%) [{symbol}]',
        fontsize=10, color=color, fontweight='bold'
    )
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Show prediction confidence distribution
max_confidences = np.max(y_pred_probs, axis=1)
correct_mask = y_pred == y_test

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confidence histogram: correct vs incorrect
axes[0].hist(max_confidences[correct_mask], bins=30, alpha=0.7,
             label=f'Correct ({correct_mask.sum()})', color='#22c55e')
axes[0].hist(max_confidences[~correct_mask], bins=30, alpha=0.7,
             label=f'Incorrect ({(~correct_mask).sum()})', color='#ef4444')
axes[0].set_title('Prediction Confidence Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Max Confidence')
axes[0].set_ylabel('Count')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Per-class confidence boxplot
class_confidences = [max_confidences[y_test == i] for i in range(NUM_CLASSES)]
bp = axes[1].boxplot(class_confidences, labels=[n.upper() for n in CLASS_NAMES],
                      patch_artist=True)
colors = ['#3b82f6', '#f59e0b', '#ef4444', '#22c55e']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
axes[1].set_title('Confidence by Defect Type', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Max Confidence')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 8. Model Export Considerations

For GCP deployment, models need to be exported in the right format:
- **SavedModel**: Standard TensorFlow format for Vertex AI endpoints
- **TF Lite**: For mobile and edge deployment
- **TF Lite + Edge TPU**: Quantized int8 for Coral Edge TPU

In [ ]:
# Export as SavedModel (for Vertex AI deployment)
EXPORT_DIR = 'defect_model_export'
model.save(os.path.join(EXPORT_DIR, 'saved_model'))
print(f"SavedModel exported to: {EXPORT_DIR}/saved_model/")

# Export as TF Lite (for edge deployment)
converter = tf.lite.TFLiteConverter.from_saved_model(
    os.path.join(EXPORT_DIR, 'saved_model')
)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

# Representative dataset for full integer quantization
def representative_dataset():
    for i in range(min(100, len(X_train))):
        yield [X_train[i:i+1].astype(np.float32)]

converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.uint8
converter.inference_output_type = tf.uint8

try:
    tflite_quant_model = converter.convert()
    tflite_path = os.path.join(EXPORT_DIR, 'defect_model_int8.tflite')
    with open(tflite_path, 'wb') as f:
        f.write(tflite_quant_model)
    
    size_mb = os.path.getsize(tflite_path) / (1024 * 1024)
    print(f"TF Lite (int8) exported to: {tflite_path}")
    print(f"Model size: {size_mb:.2f} MB")
    print(f"Edge TPU compatible: {'Yes' if size_mb < 8 else 'No (too large)'}")
except Exception as e:
    print(f"Int8 quantization note: {e}")
    print("Falling back to dynamic range quantization...")
    converter2 = tf.lite.TFLiteConverter.from_saved_model(
        os.path.join(EXPORT_DIR, 'saved_model')
    )
    converter2.optimizations = [tf.lite.Optimize.DEFAULT]
    tflite_model = converter2.convert()
    tflite_path = os.path.join(EXPORT_DIR, 'defect_model_dynamic.tflite')
    with open(tflite_path, 'wb') as f:
        f.write(tflite_model)
    size_mb = os.path.getsize(tflite_path) / (1024 * 1024)
    print(f"TF Lite (dynamic) exported to: {tflite_path}")
    print(f"Model size: {size_mb:.2f} MB")

## 9. GCP AutoML Vision API (Reference)

In production, you can use AutoML Vision as a no-code alternative.
The following shows the API calls (requires GCP project setup).

In [ ]:
# =============================================================
# GCP AutoML Vision API Reference
# (Requires GCP project with billing enabled)
# =============================================================

# --- Step 1: Upload images to Cloud Storage ---
# gsutil -m cp -r ./defect_images gs://your-bucket/defect-detection/

# --- Step 2: Create CSV import file ---
# Format: SET,GCS_PATH,LABEL
# TRAIN,gs://bucket/img001.jpg,scratch
# TRAIN,gs://bucket/img002.jpg,crack
# TEST,gs://bucket/img003.jpg,normal

# --- Step 3: Create dataset and import via SDK ---
# from google.cloud import aiplatform
#
# aiplatform.init(
#     project="your-gcp-project",
#     location="us-central1",
# )
#
# dataset = aiplatform.ImageDataset.create(
#     display_name="defect-detection-dataset",
#     gcs_source="gs://your-bucket/import.csv",
#     import_schema_uri=aiplatform.schema.dataset.ioformat
#         .image.single_label_classification,
# )

# --- Step 4: Train AutoML model ---
# job = aiplatform.AutoMLImageTrainingJob(
#     display_name="defect-automl-v1",
#     prediction_type="classification",
#     multi_label=False,
#     model_type="MOBILE_TF_LOW_LATENCY_1",  # Edge-optimized
#     budget_milli_node_hours=8000,
# )
#
# model = job.run(
#     dataset=dataset,
#     model_display_name="defect-automl-model",
#     training_fraction_split=0.8,
#     validation_fraction_split=0.1,
#     test_fraction_split=0.1,
# )

# --- Step 5: Deploy to endpoint ---
# endpoint = model.deploy(
#     deployed_model_display_name="defect-prod-v1",
#     machine_type="n1-standard-4",
#     min_replica_count=1,
#     max_replica_count=3,
# )

# --- Step 6: Make predictions ---
# import base64
# with open("test_part.jpg", "rb") as f:
#     content = base64.b64encode(f.read()).decode("utf-8")
#
# prediction = endpoint.predict(
#     instances=[{"content": content}]
# )
# print(prediction.predictions)

print("AutoML Vision API reference code (run in GCP environment with proper auth)")
print("See: https://cloud.google.com/vertex-ai/docs/image-data/classification/train-model")

## 10. Summary & Key Takeaways

| Concept | What We Did | GCP MLE Exam Relevance |
|---------|------------|------------------------|
| **Synthetic data** | Generated defect images with NumPy | Data augmentation, handling limited data |
| **Transfer learning** | MobileNetV2 with frozen backbone | When to use pre-trained models |
| **Two-phase training** | Head-only then fine-tuning | Training strategy for small datasets |
| **Class weights** | Balanced weights for imbalanced data | Handling class imbalance |
| **Per-class metrics** | Precision/recall per defect type | Choosing the right evaluation metric |
| **Model export** | SavedModel + TF Lite quantization | Edge deployment, model optimization |
| **AutoML Vision** | API reference for no-code approach | When to recommend AutoML vs custom |

### Next Steps for Production
1. Replace synthetic images with real manufacturing data
2. Add active learning pipeline for continuous improvement
3. Deploy quantized model to Edge TPU for real-time inference
4. Set up Vertex AI Model Monitoring for drift detection
5. Integrate with PLC/SCADA via OPC-UA gateway